# ViTrueVoice — Đánh Giá Inference-Only Toàn Bộ Tập Dev Seen-Only Trên Kaggle GPU (Version 4 — 16kHz Resampled Pipeline)

Notebook này thực thi quy trình suy luận độc lập (inference-only) trên toàn bộ 6.786 mẫu Dev seen-only bằng Checkpoint B (`best_model_B.pt`).
Đặc biệt: Đường đọc audio chuẩn hóa tần số lấy mẫu chính xác 16.000 Hz (`load_wave`) trước khi fit_length 64.000 mẫu, đảm bảo tính nhất quán (parity) 100% với tập huấn luyện.

### 8 Cổng Nghiệm Thu Đã Khóa (Gates 1–8):
1. **Cổng 1 — Nguồn & Hash**: Mount 2 kernel source; SHA-256 Checkpoint B (`a921c339…b56bd9f`) và Dev Manifest (`dfe82c21…f4b1f16`) khớp 100%.
2. **Cổng 2 — Dữ liệu**: Kiểm tra dung lượng; giải nén chọn lọc 1-1 (Bijective mapping) đúng 6.786 audio; 0 file thiếu, 0 file ghi đè.
3. **Cổng 3 — Mô hình**: Nạp checkpoint với `strict=True` (295/295 keys khớp, 0 missing, 0 unexpected); `eval()`, `torch.no_grad()`; không có cell train/test.
4. **Cổng 4 — Benchmark**: Đo trên GPU Kaggle thực tế (16 mẫu đại diện 4 nhóm); điều kiện nghiệm thu: $T_{\text{total}} \le 1800$s và $\ge 10$ mẫu/s.
5. **Cổng 5 — Suy luận**: Chấm 6.786 dòng, flush đệm mỗi 256 mẫu làm log tiến độ trong phiên; bắt mọi ngoại lệ đọc audio.
6. **Cổng 6 — Output**: Đủ 6.786 `utt_id` duy nhất, đúng 2.140 bona fide `base_utt`, 192 F5 `base_utt`, 0 VieNeu-TTS, 0 NaN/Inf.
7. **Cổng 7 — Phân tích**: Giữ nguyên mốc $\theta = 0.0$; tính EER/FAR/FRR kèm Bootstrap theo `base_utt` và Bootstrap 2 tầng phân tầng theo speaker.
8. **Cổng 8 — Kết luận**: Định vị 5 điểm số của `SPK_0001` bằng ECDF và phân vị; ghi nhận giới hạn phân tách của mô hình.

In [ ]:
# 1. Kiểm tra môi trường thực thi GPU & Nạp trước cấu hình HuggingFace
import os
import sys
import time
import shutil
import hashlib
import zipfile
import re
from pathlib import Path

import torch
import torchaudio
import soundfile as sf
import numpy as np
import pandas as pd
from transformers import AutoConfig, Wav2Vec2Model
from huggingface_hub import hf_hub_download

print('=' * 60)
print('⚙️ MÔI TRƯỜNG THỰC THI KAGGLE GPU')
print('=' * 60)
print(f'• Python: {sys.version}')
print(f'• PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'• GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'• Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    raise SystemError('❌ Kernel yêu cầu GPU nhưng không tìm thấy CUDA device!')

# Pre-warm HuggingFace cache với immutable commit hash & kiểm tra SHA-256 config
EXPECTED_HF_REVISION = '69e9000591623e5a4fc2f502407860bcdc0de0b2'
EXPECTED_CONFIG_SHA = 'c9aef06eaf0c2c4200132c84456db8e2a556a51d6684096f44ee3d32df9dbec5'
print(f'🌐 Đang khóa cấu hình Wav2Vec2 tại commit bất biến: {EXPECTED_HF_REVISION}...')
t0 = time.time()
try:
    cfg_file = hf_hub_download(
        repo_id='nguyenvulebinh/wav2vec2-base-vietnamese-250h',
        filename='config.json',
        revision=EXPECTED_HF_REVISION
    )
    cfg_bytes = Path(cfg_file).read_bytes()
    actual_cfg_sha = hashlib.sha256(cfg_bytes).hexdigest()
    print(f'   - Config SHA-256 thực tế:  {actual_cfg_sha}')
    print(f'   - Config SHA-256 kỳ vọng:  {EXPECTED_CONFIG_SHA}')
    assert actual_cfg_sha == EXPECTED_CONFIG_SHA, f'❌ Sai lệch SHA-256 config.json Wav2Vec2!'
    hf_cfg = AutoConfig.from_pretrained('nguyenvulebinh/wav2vec2-base-vietnamese-250h', revision=EXPECTED_HF_REVISION)
    print(f'✅ Đã khóa cấu hình bất biến và khớp SHA-256 thành công ({time.time() - t0:.2f}s)')
except Exception as e:
    raise RuntimeError(f'❌ Lỗi khi tải/khóa cấu hình HuggingFace: {e}')


In [ ]:
# 2. Khởi tạo mã nguồn ViTrueVoice trên Kaggle container
import base64
import zlib
import json
import hashlib
from pathlib import Path

PKG_B64 = '...'
EXPECTED_SRC_MANIFEST_SHA = 'c9388d1947e34550845ae473dcaaa02c615586f1c7f59f7627b13e94be99456c'

working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
pkg = json.loads(zlib.decompress(base64.b64decode(PKG_B64)).decode('utf-8'))

# Kiểm tra tính toàn vẹn của gói mã nguồn trước khi ghi
actual_manifest = {p: hashlib.sha256(content.encode('utf-8')).hexdigest() for p, content in sorted(pkg.items())}
actual_manifest_sha = hashlib.sha256(json.dumps(actual_manifest, indent=2, sort_keys=True).encode('utf-8')).hexdigest()
print(f'📦 Mã nguồn đóng gói: {len(pkg)} tệp')
print(f'   - Manifest SHA-256 thực tế:  {actual_manifest_sha}')
print(f'   - Manifest SHA-256 kỳ vọng:  {EXPECTED_SRC_MANIFEST_SHA}')
assert actual_manifest_sha == EXPECTED_SRC_MANIFEST_SHA, '❌ Sai lệch hash tham chiếu gói mã nguồn src/!'

for rel_path, content in pkg.items():
    dest = working_dir / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(content, encoding='utf-8')

if str(working_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(working_dir.resolve()))

# Kiểm tra import các module chuẩn
from src.models.ssl_aasist import SSLAASIST
from src.training.dataset import fit_length, load_wave
from src.evaluation.metrics import compute_eer, compute_det_curve

print(f'✅ Đã giải nén và xác thực toàn vẹn {len(pkg)} tệp mã nguồn vào {working_dir}!')


In [ ]:
# 3. CỔNG 1 — Xác nhận Nguồn Kernel & Kiểm tra Hash Artifacts
def compute_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

# Quét toàn bộ /kaggle/input để hiển thị danh mục tệp mount
print('📂 Đang quét danh mục /kaggle/input...')
input_p = Path('/kaggle/input')
if input_p.exists():
    all_input_files = list(input_p.rglob('*'))
    print(f'   - Tổng số mục trong /kaggle/input: {len(all_input_files)}')
    for f in all_input_files:
        if f.is_file() and any(k in f.name.lower() for k in ['best_model', 'dev', 'zip', 'manifest']):
            print(f'     • {f} ({f.stat().st_size} bytes)')

EXPECTED_CKPT_SHA = 'a921c3397946335c9543528993eafde80ac0d42decf431101635806b5c56bd9f'
EXPECTED_MANIFEST_SHA = 'dfe82c215199590fc5a90134d37f2f775307218344216ca776d47f5b4f4b1f16'

# Định vị Checkpoint B qua rglob + fallback
ckpt_candidates = (
    list(Path('/kaggle/input').rglob('*best_model_B*.pt')) +
    list(Path('/kaggle/input').rglob('*run_B*best_model*.pt')) +
    list(Path('/kaggle/input').rglob('*best_model*.pt')) +
    [
        Path('/kaggle/input/vietrue-ssl-training/best_model_B.pt'),
        Path('/kaggle/input/vietrue-ssl-training/run_B/best_model.pt'),
        Path('/kaggle/input/vietrue-ssl-training/out/best_model_B.pt'),
        Path('out/kaggle_ssl_out/best_model_B.pt'),
        Path('out/run_B/best_model.pt'),
    ]
)
ckpt_path = None
for c in ckpt_candidates:
    if c.exists() and c.is_file() and c.stat().st_size > 10000000:
        if compute_sha256(c) == EXPECTED_CKPT_SHA:
            ckpt_path = c
            break
if ckpt_path is None:
    for c in ckpt_candidates:
        if c.exists() and c.is_file() and c.stat().st_size > 10000000:
            ckpt_path = c
            break
assert ckpt_path is not None, f'❌ Không tìm thấy Checkpoint B trong các đường dẫn: {ckpt_candidates}'

# Định vị Dev Manifest (Seen-only) qua rglob + fallback
manifest_candidates = (
    list(Path('/kaggle/input').rglob('*dev.csv')) +
    [
        Path('/kaggle/input/vietrue-ssl-training/data/manifests_seen_only/dev.csv'),
        Path('/kaggle/input/vietrue-ssl-training/dev.csv'),
        Path('out/kaggle_ssl_out/data/manifests_seen_only/dev.csv'),
        Path('kaggle_v3_meta/data/manifests_seen_only/dev.csv'),
        Path('data/manifests_seen_only/dev.csv'),
    ]
)
manifest_path = None
for m in manifest_candidates:
    if m.exists() and m.is_file() and m.stat().st_size > 10000:
        if compute_sha256(m) == EXPECTED_MANIFEST_SHA:
            manifest_path = m
            break
if manifest_path is None:
    for m in manifest_candidates:
        if m.exists() and m.is_file() and m.stat().st_size > 10000:
            manifest_path = m
            break
assert manifest_path is not None, f'❌ Không tìm thấy Dev Manifest seen-only: {manifest_candidates}'

# Định vị Data ZIP qua rglob + fallback
zip_candidates = (
    list(Path('/kaggle/input').rglob('*vitruevoice*.zip')) +
    list(Path('/kaggle/input').rglob('*.zip')) +
    [
        Path('/kaggle/input/vietrue-model-training/vitruevoice_data_processed.zip'),
        Path('vitruevoice_data_processed.zip'),
    ]
)
zip_path = None
for z in zip_candidates:
    if z.exists() and z.is_file() and z.stat().st_size > 1000000:
        zip_path = z
        break
assert zip_path is not None, f'❌ Không tìm thấy ZIP dữ liệu: {zip_candidates}'

actual_ckpt_sha = compute_sha256(ckpt_path)
actual_manifest_sha = compute_sha256(manifest_path)

print(f'📦 Checkpoint: {ckpt_path}')
print(f'   - SHA-256 thực tế:  {actual_ckpt_sha}')
print(f'   - SHA-256 kỳ vọng:  {EXPECTED_CKPT_SHA}')
assert actual_ckpt_sha == EXPECTED_CKPT_SHA, f'❌ Sai lệch SHA-256 Checkpoint B!'

print(f'📄 Manifest:   {manifest_path}')
print(f'   - SHA-256 thực tế:  {actual_manifest_sha}')
print(f'   - SHA-256 kỳ vọng:  {EXPECTED_MANIFEST_SHA}')
assert actual_manifest_sha == EXPECTED_MANIFEST_SHA, f'❌ Sai lệch SHA-256 Manifest Dev!'

df_dev = pd.read_csv(manifest_path)
print(f'📊 Phân bổ Dev seen-only: {len(df_dev)} dòng')
assert len(df_dev) == 6786, f'❌ Số dòng không phải 6786: {len(df_dev)}'
assert sum(df_dev['label'] == 'bonafide') == 6210, '❌ Số dòng bonafide không phải 6210'
assert sum(df_dev['label'] == 'spoof') == 576, '❌ Số dòng spoof không phải 576'
assert 'VieNeu-TTS' not in df_dev['tts_generator'].values, '❌ Phát hiện VieNeu-TTS trong Dev seen-only!'

df_dev['base_utt'] = df_dev['utt_id'].apply(lambda x: re.sub(r'_(clean|g711|amrnb)$', '', str(x)))
bona_base = df_dev[df_dev['label'] == 'bonafide']['base_utt'].nunique()
spoof_base = df_dev[df_dev['label'] == 'spoof']['base_utt'].nunique()
print(f'   - Bonafide base_utt: {bona_base} (kỳ vọng 2140)')
print(f'   - Spoof base_utt:    {spoof_base} (kỳ vọng 192)')
assert bona_base == 2140, f'❌ Sai lệch số bonafide base_utt: {bona_base}'
assert spoof_base == 192, f'❌ Sai lệch số spoof base_utt: {spoof_base}'
print('🏁 CỔNG 1: HOÀN TOÀN ĐẠT CHUẨN!')


In [ ]:
# 4. CỔNG 2 — Dung lượng đĩa & Giải nén chọn lọc (Bijective Mapping)
t_init_start = time.perf_counter()

disk_free_gb = shutil.disk_usage(working_dir).free / (1024**3)
print(f'💾 Dung lượng đĩa trống hiện tại: {disk_free_gb:.2f} GB')
assert disk_free_gb >= 4.0, f'❌ Không đủ dung lượng đĩa tối thiểu 4.0 GB (chỉ còn {disk_free_gb:.2f} GB)!'

print(f'📦 Đang mở và kiểm định file ZIP: {zip_path} ({zip_path.stat().st_size / (1024**2):.1f} MB)...')
with zipfile.ZipFile(zip_path, 'r') as zf:
    raw_namelist = zf.namelist()
    print(f'   - Tổng số entries trong ZIP: {len(raw_namelist)}')
    
    # 1. Phát hiện trùng lặp nội bộ trong file ZIP
    assert len(raw_namelist) == len(set(raw_namelist)), '❌ Phát hiện tên entry trùng lặp trong nội bộ ZIP!'
    namelist_set = set(raw_namelist)
    
    # 2. Xây dựng ánh xạ 1-1 (Bijective mapping) từ dev.csv
    extract_targets = []
    zip_entry_to_target = {}
    target_to_zip_entry = {}
    missing_in_zip = []
    
    for idx, row in df_dev.iterrows():
        audio_rel = row['audio_path'].replace('\\', '/').lstrip('/')
        # Kiểm tra các biến thể tiền tố đường dẫn trong ZIP
        candidates_zip = [audio_rel, f'data/{audio_rel}', audio_rel.replace('data/', '')]
        found_entry = None
        for c in candidates_zip:
            if c in namelist_set:
                found_entry = c
                break
        
        if found_entry is None:
            missing_in_zip.append(audio_rel)
        else:
            dest_target = (working_dir / audio_rel).resolve()
            extract_targets.append((found_entry, dest_target))
            zip_entry_to_target[found_entry] = dest_target
            target_to_zip_entry[dest_target] = found_entry
    
    print(f'   - Đã ánh xạ: {len(extract_targets)} / {len(df_dev)} tệp')
    if missing_in_zip:
        print(f'❌ Thiếu {len(missing_in_zip)} tệp trong ZIP! 5 mẫu đầu: {missing_in_zip[:5]}')
        raise FileNotFoundError(f'Thiếu {len(missing_in_zip)} tệp audio trong ZIP!')
        
    # 3. Khẳng định không có 2 entry ZIP nào cùng ghi đè một target path
    assert len(set(extract_targets)) == len(df_dev), '❌ Phát hiện trùng lặp target path trong ánh xạ!'
    assert len(target_to_zip_entry) == len(df_dev), '❌ Phát hiện nhiều entry ZIP cùng ghi đè một file đích!'
    
    # 4. Thực hiện giải nén chọn lọc đúng 6786 file cần thiết
    print('🚀 Đang thực hiện Selective Unzip đúng 6786 file audio...')
    t_unzip0 = time.perf_counter()
    for zip_entry, dest_target in extract_targets:
        dest_target.parent.mkdir(parents=True, exist_ok=True)
        if not dest_target.exists():
            with zf.open(zip_entry) as src, open(dest_target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
    print(f'✅ Giải nén chọn lọc hoàn tất trong {time.perf_counter() - t_unzip0:.1f}s!')

# 5. Quét toàn bộ hệ thống file để bảo đảm 100% tệp tồn tại và đọc được
missing_files = []
for idx, row in df_dev.iterrows():
    p = working_dir / row['audio_path'].replace('\\', '/')
    if not p.is_file():
        missing_files.append(str(p))

print(f'🔍 Quét kiểm tra 6786 đường dẫn audio trên đĩa: {len(df_dev) - len(missing_files)} / {len(df_dev)} tồn tại')
assert len(missing_files) == 0, f'❌ Vẫn còn {len(missing_files)} tệp không tìm thấy sau khi giải nén!'
print('🏁 CỔNG 2: HOÀN TOÀN ĐẠT CHUẨN!')


In [ ]:
# 5. CỔNG 3 — Nạp Checkpoint Chế độ strict=True & Khóa trạng thái eval-only
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'🤖 Đang nạp Checkpoint B từ: {ckpt_path}...')
ckpt_dict = torch.load(ckpt_path, map_location=device, weights_only=False)
ckpt_meta = {
    'branch': ckpt_dict.get('branch'),
    'phase': ckpt_dict.get('phase'),
    'epoch': ckpt_dict.get('epoch'),
    'dev_eer': ckpt_dict.get('dev_eer') or ckpt_dict.get('val_eer'),
}
print(f'   - Metadata: Nhánh {ckpt_meta["branch"]} | Phase {ckpt_meta["phase"]} | Epoch {ckpt_meta["epoch"]} | Dev EER lúc train: {ckpt_meta["dev_eer"] * 100:.2f}%')

# Khởi tạo kiến trúc SSL-AASIST theo đúng kwargs đã lưu và nạp hf_cfg đã khóa
model_kwargs = dict(ckpt_dict.get('model_kwargs', {}))
model_kwargs['ssl_config'] = hf_cfg
print(f'   - Khởi tạo kiến trúc với model_kwargs (offline config): {list(model_kwargs.keys())}')
model = SSLAASIST(**model_kwargs)

# Nạp state_dict ở chế độ strict=True
print('🔒 Đang nạp trọng số ở chế độ strict=True...')
state_dict = ckpt_dict['model']
print(f'   - Số lượng keys trong checkpoint: {len(state_dict)}')
assert len(state_dict) == 295, f'❌ Số lượng keys trong checkpoint không phải 295 (thực tế {len(state_dict)})!'

# load_state_dict strict sẽ tự động ném RuntimeError nếu có missing_keys hoặc unexpected_keys
load_result = model.load_state_dict(state_dict, strict=True)
print(f'✅ Khớp toàn bộ 295/295 keys! {load_result}')

# Khóa toàn bộ tham số, đặt eval mode
model.to(device)
model.eval()
for param in model.parameters():
    param.requires_grad = False

assert not model.training, '❌ Mô hình đang không ở chế độ eval()!'
print('🏁 CỔNG 3: HOÀN TOÀN ĐẠT CHUẨN (strict=True, eval-only)!')


In [ ]:
# 6. CỔNG 4 — Benchmark Ngắn trên GPU Kaggle & Tiêu chí nghiệm thu định lượng
from torch.utils.data import Dataset, DataLoader

# Dataset suy luận tối ưu bộ nhớ
class DevEvalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root: Path):
        self.df = df.reset_index(drop=True)
        self.root = root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = self.root / row['audio_path'].replace('\\', '/')
        # Chuẩn hóa tần số lấy mẫu 16.000 Hz đồng bộ hoàn toàn với ViTrueVoiceDataset
        wav = load_wave(p, sr=16000)
        # Cố định chính xác 64.000 mẫu bằng repeat padding
        wav = fit_length(wav, 64000, False, np.random.default_rng(0))
        return {
            'idx': idx,
            'utt_id': str(row['utt_id']),
            'speaker_id': str(row['speaker_id']),
            'base_utt': str(row['base_utt']),
            'label': str(row['label']),
            'codec': str(row['codec']),
            'source': str(row.get('source', '')),
            'wav': torch.from_numpy(wav).float(),
        }

# Chọn 16 mẫu đại diện cho 4 điều kiện: Bona clean, F5 clean, F5 G.711, F5 AMR-NB
benchmark_samples = pd.concat([
    df_dev[(df_dev['label'] == 'bonafide') & (df_dev['codec'] == 'clean_16khz')].head(4),
    df_dev[(df_dev['label'] == 'spoof') & (df_dev['codec'] == 'clean_16khz')].head(4),
    df_dev[(df_dev['label'] == 'spoof') & (df_dev['codec'] == 'g711_alaw')].head(4),
    df_dev[(df_dev['label'] == 'spoof') & (df_dev['codec'] == 'amr_nb')].head(4),
], ignore_index=True)

print(f'🧪 Chạy benchmark thử nghiệm trên {len(benchmark_samples)} mẫu đại diện 4 nhóm...')
bm_ds = DevEvalDataset(benchmark_samples, working_dir)
bm_loader = DataLoader(bm_ds, batch_size=16, shuffle=False)

# 1. Warm-up pass với 4 mẫu để nạp CUDA kernels và khởi động JIT pipeline
warmup_samples = benchmark_samples.head(4)
warmup_ds = DevEvalDataset(warmup_samples, working_dir)
warmup_loader = DataLoader(warmup_ds, batch_size=4, shuffle=False)
with torch.inference_mode():
    for wb in warmup_loader:
        _ = model.score(wb['wav'].to(device))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
print('🔥 Warm-up GPU hoàn tất. Bắt đầu đo lường chuẩn xác...')

# 2. Đo lường throughput ổn định với đồng bộ CUDA chặt chẽ
torch.cuda.reset_peak_memory_stats()
if torch.cuda.is_available():
    torch.cuda.synchronize()

t_bm0 = time.perf_counter()
with torch.inference_mode():
    for b in bm_loader:
        _ = model.score(b['wav'].to(device))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
dt_bm = time.perf_counter() - t_bm0

vram_peak_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
throughput = len(benchmark_samples) / dt_bm
t_init = time.perf_counter() - t_init_start
t_infer_est = len(df_dev) / throughput
t_total_est = t_init + t_infer_est

print(f'📊 KẾT QUẢ ĐO BENCHMARK THỰC TẾ TRÊN GPU {torch.cuda.get_device_name(0)}:')
print(f'   - Thời gian khởi tạo & giải nén (T_init): {t_init:.1f}s')
print(f'   - VRAM cấp phát đỉnh:                     {vram_peak_mb:.1f} MB')
print(f'   - Tốc độ suy luận thực tế:                {throughput:.1f} mẫu/giây')
print(f'   - Thời gian suy luận dự tính (6786 mẫu):  {t_infer_est / 60:.1f} phút ({t_infer_est:.1f}s)')
print(f'   - Tổng thời gian ước tính toàn bộ job:    {t_total_est / 60:.1f} phút ({t_total_est:.1f}s)')

# Kiểm tra tiêu chuẩn định lượng đã chốt
assert throughput >= 10.0, f'❌ Tốc độ suy luận quá chậm ({throughput:.1f} mẫu/s < 10 mẫu/s)!'
assert t_total_est <= 1800.0, f'❌ Tổng thời gian ước tính vượt quá giới hạn 30 phút ({t_total_est:.1f}s > 1800s)!'
print('🏁 CỔNG 4: HOÀN TOÀN ĐẠT CHUẨN!')


In [ ]:
# 7. CỔNG 5 & 6 — Suy luận toàn bộ 6786 dòng Dev seen-only & Kiểm tra Output
eval_ds = DevEvalDataset(df_dev, working_dir)
eval_loader = DataLoader(eval_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

records = []
temp_csv = working_dir / 'dev_scores_partial.csv'
out_csv = working_dir / 'dev_scores_full.csv'

print(f'▶️ Bắt đầu suy luận toàn bộ {len(eval_ds)} dòng (batch_size=16, flush mỗi 256 mẫu)...')
t_infer_start = time.perf_counter()
chunk_counter = 0

with torch.inference_mode():
    for batch in eval_loader:
        wavs = batch['wav'].to(device, non_blocking=True)
        scores = model.score(wavs).cpu().numpy()
        
        for i in range(len(scores)):
            s = float(scores[i])
            records.append({
                'utt_id': batch['utt_id'][i],
                'speaker_id': batch['speaker_id'][i],
                'base_utt': batch['base_utt'][i],
                'label': batch['label'][i],
                'codec': batch['codec'][i],
                'source': batch['source'][i],
                'raw_score': s,
                'prediction_thr0': 'bonafide' if s >= 0.0 else 'spoof',
                'checkpoint_sha256': actual_ckpt_sha,
                'manifest_sha256': actual_manifest_sha,
            })
            chunk_counter += 1
            
        # Flush tiến độ mỗi 256 mẫu làm log
        if chunk_counter >= 256:
            pd.DataFrame(records).to_csv(temp_csv, index=False)
            elapsed = time.perf_counter() - t_infer_start
            speed = len(records) / elapsed if elapsed > 0 else 0
            print(f'  💾 Đã chấm {len(records)}/{len(eval_ds)} mẫu ({speed:.1f} mẫu/s)...', flush=True)
            chunk_counter = 0

total_infer_time = time.perf_counter() - t_infer_start
print(f'🏁 Hoàn thành chấm toàn bộ {len(records)} mẫu trong {total_infer_time:.1f}s ({len(records)/total_infer_time:.1f} mẫu/s)!')

df_out = pd.DataFrame(records)
df_out.to_csv(out_csv, index=False)
if temp_csv.exists():
    temp_csv.unlink()

# CỔNG 6 — Kiểm tra toàn vẹn Output
print('=' * 60)
print('🔍 KIỂM TRA TOÀN VẸN ĐẦU RA (CỔNG 6)')
print('=' * 60)
assert len(df_out) == 6786, f'❌ Số dòng không phải 6786: {len(df_out)}'
assert df_out['utt_id'].nunique() == 6786, '❌ Có utt_id bị trùng lặp!'
assert df_out['raw_score'].isna().sum() == 0, '❌ Phát hiện điểm số NaN!'
assert np.isinf(df_out['raw_score']).sum() == 0, '❌ Phát hiện điểm số vô hạn (Inf)!'

n_bona = (df_out['label'] == 'bonafide').sum()
n_spoof = (df_out['label'] == 'spoof').sum()
assert n_bona == 6210, f'❌ Số dòng bona fide sai lệch: {n_bona}'
assert n_spoof == 576, f'❌ Số dòng spoof sai lệch: {n_spoof}'

u_bona = df_out[df_out['label'] == 'bonafide']['base_utt'].nunique()
u_spoof = df_out[df_out['label'] == 'spoof']['base_utt'].nunique()
assert u_bona == 2140, f'❌ Base utterance bona fide sai lệch: {u_bona}'
assert u_spoof == 192, f'❌ Base utterance spoof sai lệch: {u_spoof}'

print(f'✅ File kết quả đã ghi: {out_csv} ({out_csv.stat().st_size / 1024:.1f} KB)')
print('🏁 CỔNG 5 & 6: HOÀN TOÀN ĐẠT CHUẨN!')


In [ ]:
# 8. CỔNG 7 — Phân tích Thống kê Toàn diện & Bootstrap 2 Tầng Phân tầng
print('=' * 60)
print('📈 PHÂN TÍCH THỐNG KÊ TOÀN BỘ DEV SEEN-ONLY (CỔNG 7)')
print('=' * 60)

# 1. Điểm EER toàn tập & tại ngưỡng chuẩn 0.0
scores_all = df_out['raw_score'].to_numpy()
labels_all = (df_out['label'] == 'bonafide').astype(int).to_numpy()
eer_all, eer_thr = compute_eer(scores_all, labels_all)

bona_scores = df_out[df_out['label'] == 'bonafide']['raw_score']
spoof_scores = df_out[df_out['label'] == 'spoof']['raw_score']
frr_thr0 = (bona_scores < 0.0).mean() * 100.0
far_thr0 = (spoof_scores >= 0.0).mean() * 100.0

print(f'• EER Toàn tập:       {eer_all * 100:.2f}% (tại ngưỡng EER = {eer_thr:+.4f})')
print(f'• Tại ngưỡng θ = 0.0: FRR = {frr_thr0:.2f}% (Bona bị loại) | FAR = {far_thr0:.2f}% (Spoof bị lọt)')

# 2. Phân tầng theo Codec tại ngưỡng 0.0
print('\n📊 Bảng Phân Tầng Theo Codec:')
codec_stats = []
for c in ['clean_16khz', 'g711_alaw', 'amr_nb']:
    sub = df_out[df_out['codec'] == c]
    if len(sub) == 0:
        continue
    s_c = sub['raw_score'].to_numpy()
    l_c = (sub['label'] == 'bonafide').astype(int).to_numpy()
    e_c, t_c = compute_eer(s_c, l_c) if len(np.unique(l_c)) > 1 else (np.nan, np.nan)
    frr_c = (sub[sub['label'] == 'bonafide']['raw_score'] < 0.0).mean() * 100.0
    far_c = (sub[sub['label'] == 'spoof']['raw_score'] >= 0.0).mean() * 100.0
    codec_stats.append({
        'Codec': c,
        'N_Bona': sum(sub['label'] == 'bonafide'),
        'N_Spoof': sum(sub['label'] == 'spoof'),
        'EER (%)': e_c * 100 if not np.isnan(e_c) else np.nan,
        'FRR_θ0 (%)': frr_c,
        'FAR_θ0 (%)': far_c,
    })
df_codec = pd.DataFrame(codec_stats)
print(df_codec.to_string(index=False))

# 3. Bootstrap theo base_utt (B = 1000)
print('\n⏳ Đang chạy Clustered Bootstrap theo base_utt (B = 1000)...')
rng = np.random.default_rng(42)
unique_base_utts = df_out['base_utt'].unique()
base_utt_to_indices = df_out.groupby('base_utt').indices

B = 1000
boot_eers_utt = []
boot_fars_utt = []
boot_frrs_utt = []

for _ in range(B):
    sampled_utts = rng.choice(unique_base_utts, size=len(unique_base_utts), replace=True)
    sampled_idx = np.concatenate([base_utt_to_indices[u] for u in sampled_utts])
    b_scores = scores_all[sampled_idx]
    b_labels = labels_all[sampled_idx]
    if len(np.unique(b_labels)) > 1:
        e, _ = compute_eer(b_scores, b_labels)
        boot_eers_utt.append(e)
        boot_frrs_utt.append((b_scores[b_labels == 1] < 0.0).mean() * 100.0)
        boot_fars_utt.append((b_scores[b_labels == 0] >= 0.0).mean() * 100.0)

eer_ci_utt = (np.percentile(boot_eers_utt, 2.5) * 100, np.percentile(boot_eers_utt, 97.5) * 100)
frr_ci_utt = (np.percentile(boot_frrs_utt, 2.5), np.percentile(boot_frrs_utt, 97.5))
far_ci_utt = (np.percentile(boot_fars_utt, 2.5), np.percentile(boot_fars_utt, 97.5))
print(f'• 95% CI Bootstrap theo base_utt: EER = [{eer_ci_utt[0]:.2f}%, {eer_ci_utt[1]:.2f}%] | FRR_θ0 = [{frr_ci_utt[0]:.2f}%, {frr_ci_utt[1]:.2f}%] | FAR_θ0 = [{far_ci_utt[0]:.2f}%, {far_ci_utt[1]:.2f}%]')

# 4. Bootstrap 2 Tầng Phân Tầng: Speaker -> Base Utterance (B = 1000)
print('⏳ Đang chạy Stratified Clustered Two-Stage Bootstrap (Speaker -> Base Utterance, B = 1000)...')
speakers = df_out['speaker_id'].unique()
spk_bona_utts = df_out[df_out['label'] == 'bonafide'].groupby('speaker_id')['base_utt'].unique().to_dict()
spk_spoof_utts = df_out[df_out['label'] == 'spoof'].groupby('speaker_id')['base_utt'].unique().to_dict()

boot_eers_2s = []
boot_fars_2s = []
boot_frrs_2s = []

for _ in range(B):
    # Tầng 1: Resample 15 speaker có hoàn lại (mỗi replica là 1 cluster riêng)
    sampled_spks = rng.choice(speakers, size=len(speakers), replace=True)
    sampled_all_idx = []
    
    for s in sampled_spks:
        # Tầng 2: Resample bona fide base_utt
        b_utts = spk_bona_utts.get(s, np.array([]))
        if len(b_utts) > 0:
            chosen_b = rng.choice(b_utts, size=len(b_utts), replace=True)
            for u in chosen_b:
                sampled_all_idx.extend(base_utt_to_indices[u])
        # Tầng 2: Resample spoof base_utt (kéo theo 3 codec)
        s_utts = spk_spoof_utts.get(s, np.array([]))
        if len(s_utts) > 0:
            chosen_s = rng.choice(s_utts, size=len(s_utts), replace=True)
            for u in chosen_s:
                sampled_all_idx.extend(base_utt_to_indices[u])
                
    b2_scores = scores_all[sampled_all_idx]
    b2_labels = labels_all[sampled_all_idx]
    if len(np.unique(b2_labels)) > 1:
        e2, _ = compute_eer(b2_scores, b2_labels)
        boot_eers_2s.append(e2)
        boot_frrs_2s.append((b2_scores[b2_labels == 1] < 0.0).mean() * 100.0)
        boot_fars_2s.append((b2_scores[b2_labels == 0] >= 0.0).mean() * 100.0)

eer_ci_2s = (np.percentile(boot_eers_2s, 2.5) * 100, np.percentile(boot_eers_2s, 97.5) * 100)
frr_ci_2s = (np.percentile(boot_frrs_2s, 2.5), np.percentile(boot_frrs_2s, 97.5))
far_ci_2s = (np.percentile(boot_fars_2s, 2.5), np.percentile(boot_fars_2s, 97.5))
print(f'• 95% CI Bootstrap 2 Tầng:        EER = [{eer_ci_2s[0]:.2f}%, {eer_ci_2s[1]:.2f}%] | FRR_θ0 = [{frr_ci_2s[0]:.2f}%, {frr_ci_2s[1]:.2f}%] | FAR_θ0 = [{far_ci_2s[0]:.2f}%, {far_ci_2s[1]:.2f}%]')
print('🏁 CỔNG 7: HOÀN TOÀN ĐẠT CHUẨN!')


In [ ]:
# 9. CỔNG 8 — Định vị 5 Điểm Số SPK_0001 trên Phân Phối Toàn Dev & Kết Luận Giới Hạn
print('=' * 60)
print('🎯 ĐỊNH VỊ 5 CA SPK_0001 TRÊN PHÂN PHỐI DEV (CỔNG 8)')
print('=' * 60)

spk_0001_samples = [
    {'file': 'SPK_0001_01.wav', 'sha256': '51c9f06b9043', 'score': -11.9034},
    {'file': 'SPK_0001_02.wav', 'sha256': '10fc5c876faf', 'score': -14.3603},
    {'file': 'SPK_0001_03.wav', 'sha256': '2652a954094f', 'score': -21.7787},
    {'file': 'SPK_0001_04.wav', 'sha256': '48b542e34cac', 'score': -10.2416},
    {'file': 'SPK_0001_05.wav', 'sha256': '0e8816b4a712', 'score': -15.9917},
]
spk_0001_scores = [item['score'] for item in spk_0001_samples]
spk_0001_median = float(np.median(spk_0001_scores))

bona_all_scores = df_out[df_out['label'] == 'bonafide']['raw_score'].to_numpy()
spoof_all_scores = df_out[df_out['label'] == 'spoof']['raw_score'].to_numpy()

# Hàm tính ECDF và phân vị
def get_percentile(arr, val):
    return (arr <= val).mean() * 100.0

print('Bảng So Sánh Phân Vị (ECDF) của 5 phát ngôn SPK_0001:')
comp_data = []
for item in spk_0001_samples:
    s = item['score']
    pct_bona = get_percentile(bona_all_scores, s)
    pct_spoof = get_percentile(spoof_all_scores, s)
    comp_data.append({
        'Tệp SPK_0001': item['file'],
        'SHA-256 (12 ký tự)': item['sha256'],
        'Raw Score': f'{s:.4f}',
        'Phân vị trong Bona fide (%)': f'{pct_bona:.3f}%',
        'Phân vị trong Spoof F5 (%)':   f'{pct_spoof:.3f}%',
    })
print(pd.DataFrame(comp_data).to_string(index=False))

# Phân tích tương quan các người nói Bona fide trong Dev
print('\n📊 Thống kê Điểm Số Bona fide Theo Từng Người Nói (15 Speaker):')
spk_summary = []
for spk, grp in df_out[df_out['label'] == 'bonafide'].groupby('speaker_id'):
    s_vals = grp['raw_score'].to_numpy()
    spk_summary.append({
        'Speaker': spk,
        'N_Bona': len(s_vals),
        'Mean': np.mean(s_vals),
        'Median': np.median(s_vals),
        'Min': np.min(s_vals),
        'Max': np.max(s_vals),
        'Tỷ lệ < 0 (%)': (s_vals < 0.0).mean() * 100.0,
    })
df_spk = pd.DataFrame(spk_summary)
print(df_spk.to_string(index=False))

# Đóng gói tóm tắt JSON
summary_json = {
    'checkpoint': actual_ckpt_sha,
    'manifest': actual_manifest_sha,
    'total_samples': len(df_out),
    'eer': float(eer_all),
    'eer_threshold': float(eer_thr),
    'frr_thr0': float(frr_thr0),
    'far_thr0': float(far_thr0),
    'ci_base_utt': {
        'eer': [float(eer_ci_utt[0]), float(eer_ci_utt[1])],
        'frr_thr0': [float(frr_ci_utt[0]), float(frr_ci_utt[1])],
        'far_thr0': [float(far_ci_utt[0]), float(far_ci_utt[1])],
    },
    'ci_two_stage': {
        'eer': [float(eer_ci_2s[0]), float(eer_ci_2s[1])],
        'frr_thr0': [float(frr_ci_2s[0]), float(frr_ci_2s[1])],
        'far_thr0': [float(far_ci_2s[0]), float(far_ci_2s[1])],
    },
    'spk_0001_analysis': {
        'scores': spk_0001_scores,
        'median': spk_0001_median,
        'median_percentile_bona': float(get_percentile(bona_all_scores, spk_0001_median)),
        'median_percentile_spoof': float(get_percentile(spoof_all_scores, spk_0001_median)),
    }
}

with open(working_dir / 'dev_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary_json, f, indent=2, ensure_ascii=False)

print('\n🏁 CỔNG 8: HOÀN TẤT ĐỊNH VỊ VÀ ĐÓNG GÓI BÁO CÁO!')
print(f'✅ Kết quả đã lưu tại: {working_dir / "dev_scores_full.csv"} và {working_dir / "dev_summary.json"}')
